In [0]:
%sql
USE CATALOG mvp_pipeline;

##2. Dimensão de tempo

Grão: um mês. Cobre 2015 a 2026, intervalo que contém todas as séries.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_tempo AS
WITH meses AS (
  SELECT explode(sequence(to_date('2015-01-01'), to_date('2026-12-01'), INTERVAL 1 MONTH)) AS data_ref
),
anos_completos AS (
  SELECT ano FROM silver.icms_cnae_subclasse
  GROUP BY ano HAVING count(DISTINCT mes) = 12
)
SELECT
  CAST(date_format(data_ref, 'yyyyMM') AS INT)      AS sk_tempo,
  data_ref,
  year(data_ref)                                    AS ano,
  quarter(data_ref)                                 AS trimestre,
  month(data_ref)                                   AS mes,
  CASE month(data_ref)
    WHEN 1 THEN 'Janeiro'   WHEN 2 THEN 'Fevereiro' WHEN 3 THEN 'Março'
    WHEN 4 THEN 'Abril'     WHEN 5 THEN 'Maio'      WHEN 6 THEN 'Junho'
    WHEN 7 THEN 'Julho'     WHEN 8 THEN 'Agosto'    WHEN 9 THEN 'Setembro'
    WHEN 10 THEN 'Outubro'  WHEN 11 THEN 'Novembro' ELSE 'Dezembro'
  END                                               AS nome_mes,
  year(data_ref) IN (SELECT ano FROM anos_completos) AS ano_completo,
  current_timestamp()                               AS _processamento_ts
FROM meses;

In [0]:
%sql

-- Verificação

SELECT ano, count(*) AS meses, max(ano_completo) AS ano_completo
FROM gold.dim_tempo GROUP BY ano ORDER BY ano;

##3. Dimensão de atividade econômica

Grão: uma subclasse CNAE de sete dígitos. Reúne todos os códigos que aparecem em qualquer fonte

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_cnae AS
WITH codigos AS (
  SELECT DISTINCT cnae_subclasse FROM silver.icms_cnae_subclasse
  UNION SELECT DISTINCT cnae_subclasse FROM silver.desoneracoes WHERE cnae_subclasse IS NOT NULL
  UNION SELECT DISTINCT cnae_subclasse FROM silver.cadastro_setor
  UNION SELECT DISTINCT cnae_subclasse FROM silver.cnae_cadeia
),
nome_icms AS (
  SELECT cnae_subclasse,
         max(nome_cnae_subclasse) AS nome,
         max(versao_cnae)         AS versao
  FROM silver.icms_cnae_subclasse GROUP BY cnae_subclasse
),
nome_cadeia AS (
  SELECT cnae_subclasse, max(nome_cnae_subclasse) AS nome
  FROM silver.cnae_cadeia GROUP BY cnae_subclasse
),
hierarquia AS (
  SELECT cnae_subclasse,
         max(cnae_secao)   AS cnae_secao,
         max(cnae_divisao) AS cnae_divisao,
         max(cnae_grupo)   AS cnae_grupo,
         max(cnae_classe)  AS cnae_classe
  FROM silver.desoneracoes
  WHERE nivel_agregacao = 'DETALHADO' AND cnae_subclasse IS NOT NULL
  GROUP BY cnae_subclasse
)
SELECT
  c.cnae_subclasse,
  coalesce(ni.nome, nc.nome)                        AS nome_cnae_subclasse,
  ni.versao                                         AS versao_cnae,
  h.cnae_secao, h.cnae_divisao, h.cnae_grupo, h.cnae_classe,
  c.cnae_subclasse = '0000000'                      AS sem_cnae,
  nc.nome IS NOT NULL                               AS tem_cadeia,
  current_timestamp()                               AS _processamento_ts
FROM codigos c
LEFT JOIN nome_icms   ni ON ni.cnae_subclasse = c.cnae_subclasse
LEFT JOIN nome_cadeia nc ON nc.cnae_subclasse = c.cnae_subclasse
LEFT JOIN hierarquia  h  ON h.cnae_subclasse  = c.cnae_subclasse;

In [0]:
%sql

SELECT * FROM gold.dim_cnae

##4. dimensão de cadeia produtiva

Grão: uma cadeia. As ressalvas de leitura medidas no perfil de qualidade viram atributos da dimensão

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_cadeia_produtiva AS
WITH cnaes AS (
  SELECT cadeia, cadeia_prioritaria, count(DISTINCT cnae_subclasse) AS qtd_cnaes
  FROM silver.cnae_cadeia GROUP BY cadeia, cadeia_prioritaria
),
cobertura AS (
  SELECT c.cadeia,
         round(100 * count(DISTINCT i.cnae_subclasse) / count(DISTINCT c.cnae_subclasse), 1) AS cobertura_icms_perc
  FROM silver.cnae_cadeia c
  LEFT JOIN (SELECT DISTINCT cnae_subclasse FROM silver.icms_cnae_subclasse WHERE ano = 2024) i
    ON i.cnae_subclasse = c.cnae_subclasse
  GROUP BY c.cadeia
)
SELECT
  n.cadeia,
  n.cadeia_prioritaria,
  n.qtd_cnaes,
  b.cobertura_icms_perc,
  n.qtd_cnaes < 5                  AS ressalva_volatilidade,
  b.cobertura_icms_perc < 70       AS ressalva_cobertura,
  CASE
    WHEN n.qtd_cnaes < 5 AND b.cobertura_icms_perc < 70
      THEN 'Poucas subclasses e cobertura parcial de ICMS'
    WHEN n.qtd_cnaes < 5
      THEN 'Definida por menos de cinco subclasses: série sensível a um único contribuinte'
    WHEN b.cobertura_icms_perc < 70
      THEN 'Cadeia majoritariamente de serviço ou produção primária: o ICMS mede fração dela'
    ELSE NULL
  END                              AS ressalva,
  current_timestamp()              AS _processamento_ts
FROM cnaes n JOIN cobertura b ON b.cadeia = n.cadeia;

In [0]:
%sql

SELECT * FROM gold.dim_cadeia_produtiva

##4. ponte CNAE x cadeia produtiva

36 subclasses pertencem a duas cadeias. A ponte é o que permite o recorte setorial sem duplicar o fato.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.ponte_cnae_cadeia AS
SELECT DISTINCT
  cnae_subclasse,
  cadeia,
  cadeia_prioritaria,
  current_timestamp() AS _processamento_ts
FROM silver.cnae_cadeia;

In [0]:
%sql
-- subclasses em mais de uma cadeia: esperado 36 entre as prioritárias
SELECT count(*) AS cnaes_em_duas_ou_mais FROM (
  SELECT cnae_subclasse FROM gold.ponte_cnae_cadeia WHERE cadeia_prioritaria
  GROUP BY cnae_subclasse HAVING count(DISTINCT cadeia) > 1);

##6. Dimensão de município

Grão: um município do IBGE, mais os dois códigos residuais da SEFAZ

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_municipio AS
WITH corede_por_municipio AS (
  SELECT cod_municipio_ibge, max(nome_corede) AS nome_corede
  FROM silver.cadastro_municipio
  WHERE nome_corede IS NOT NULL AND upper(nome_corede) <> 'SEM COREDE'
  GROUP BY cod_municipio_ibge
)
SELECT
  m.cod_municipio_ibge,
  m.nome_municipio,
  c.nome_corede,
  m.microrregiao,
  m.mesorregiao,
  d.cod_munic_sefaz,
  FALSE                AS pseudo_municipio,
  current_timestamp()  AS _processamento_ts
FROM silver.municipio_ibge m
LEFT JOIN corede_por_municipio c ON c.cod_municipio_ibge = m.cod_municipio_ibge
LEFT JOIN silver.depara_municipio d ON d.cod_municipio_ibge = m.cod_municipio_ibge
UNION ALL
SELECT
  CAST(NULL AS STRING), nome_municipio, CAST(NULL AS STRING),
  CAST(NULL AS STRING), CAST(NULL AS STRING), cod_munic_sefaz,
  TRUE, current_timestamp()
FROM silver.depara_municipio WHERE pseudo_municipio;

##7. dimensão de benefício fiscal

Grão: um dispositivo, identificado por imposto + tipo_beneficio + cod_beneficio

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_beneficio AS
SELECT
  imposto,
  tipo_beneficio,
  cod_beneficio,
  max(descr_beneficio)  AS descr_beneficio,
  max(legislacao)       AS legislacao,
  max(finalidade)       AS finalidade,
  max(justificativa)    AS justificativa,

  -- regra 2: o que a Receita Estadual soma ao total do Estado
  NOT (upper(tipo_beneficio) IN ('IMUNIDADE', 'NAO INCIDÊNCIA', 'SIMPLES NACIONAL')
       OR upper(max(finalidade)) = 'NEC EXPORTAÇÕES')      AS integra_total_estadual,

  -- regra 4: só finalidade econômica sustenta leitura setorial
  upper(max(finalidade)) = 'ECONÔMICO'                     AS finalidade_economica,

  -- regra 3: dispositivos criados para a calamidade de 2024
  (imposto = 'ICMS' AND
   ((tipo_beneficio = 'ISENÇÃO'           AND cod_beneficio IN (188, 189, 192, 194))
     OR (tipo_beneficio = 'CRÉDITO PRESUMIDO' AND cod_beneficio = 235)))
                                                           AS evento_extraordinario,

  -- origem do valor: declarado pelo contribuinte ou estimado pela Receita
  CASE WHEN upper(tipo_beneficio) LIKE '%PRESUMIDO%'
       THEN 'DIRETO' ELSE 'ESTIMATIVA' END                 AS origem_valor,

  current_timestamp() AS _processamento_ts
FROM silver.desoneracoes
GROUP BY imposto, tipo_beneficio, cod_beneficio;

##8. dimensão de categoria de contribuinte
Separa três conceitos que a fonte mistura: porte empresarial, produtor rural e elegibilidade para análise

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_categoria_contribuinte AS
SELECT DISTINCT
  categoria,
  upper(categoria) IN ('MEI', 'SIMPLES NACIONAL')       AS e_mpe,
  upper(categoria) IN ('MICROPRODUTOR', 'PRODUTOR')     AS e_produtor_rural,
  upper(categoria) <> 'MEI'                             AS entra_analise,
  CASE WHEN upper(categoria) = 'MEI'
       THEN 'Baixas acumuladas desde set/2024, contrariando o dicionário da fonte; série não comparável'
       ELSE NULL END                                    AS ressalva,
  current_timestamp()                                   AS _processamento_ts
FROM silver.cadastro_setor;

##9. fato de ICMS por atividade econômica
Grão: mês × versão da CNAE × subclasse × nome da categoria residual

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fato_icms_cnae AS
SELECT
  CAST(concat(i.ano, lpad(i.mes, 2, '0')) AS INT) AS sk_tempo,
  i.cnae_subclasse,
  i.versao_cnae,
  i.nome_cnae_subclasse,
  i.sem_cnae,
  i.valor_icms,
  current_timestamp() AS _processamento_ts
FROM silver.icms_cnae_subclasse i;

##10. fato de arrecadação por território
Grão: mês × município × tributo.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fato_arrecadacao_municipio AS
SELECT
  CAST(concat(a.ano, lpad(a.mes, 2, '0')) AS INT) AS sk_tempo,
  d.cod_municipio_ibge,
  a.cod_munic_sefaz,
  d.pseudo_municipio,
  a.tributo,
  a.valor_arrecadado,
  current_timestamp() AS _processamento_ts
FROM silver.arrecadacao_municipio a
LEFT JOIN silver.depara_municipio d ON d.cod_munic_sefaz = a.cod_munic_sefaz;

##11. fatos de desoneração
Dois fatos de granularidades diferentes

In [0]:
%sql
-- nível detalhado: ano x dispositivo x COREDE x subclasse
CREATE OR REPLACE TABLE gold.fato_desoneracao_detalhada AS
SELECT
  d.ano,
  t.ano_completo,
  d.imposto, d.tipo_beneficio, d.cod_beneficio,
  d.nome_corede,
  d.cnae_subclasse,
  d.valor_desonerado,
  d.qtd_empresas,
  current_timestamp() AS _processamento_ts
FROM silver.desoneracoes d
LEFT JOIN (SELECT DISTINCT ano, ano_completo FROM gold.dim_tempo) t ON t.ano = d.ano
WHERE d.nivel_agregacao = 'DETALHADO';

In [0]:
%sql
-- nível agregado estadual: ano x dispositivo, sem chave setorial nem territorial
CREATE OR REPLACE TABLE gold.fato_desoneracao_agregada AS
SELECT
  d.ano,
  t.ano_completo,
  d.imposto, d.tipo_beneficio, d.cod_beneficio,
  d.valor_desonerado,
  d.qtd_empresas,
  current_timestamp() AS _processamento_ts
FROM silver.desoneracoes d
LEFT JOIN (SELECT DISTINCT ano, ano_completo FROM gold.dim_tempo) t ON t.ano = d.ano
WHERE d.nivel_agregacao = 'AGREGADO_ESTADUAL';

##12. fatos de cadastro de contribuintes
por atividade econômica e por território.

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fato_cadastro_setor AS
SELECT
  CAST(concat(c.ano, lpad(c.mes, 2, '0')) AS INT) AS sk_tempo,
  c.categoria,
  c.cnae_subclasse,
  c.setor, c.area, c.atividade,
  c.qtd_ativos,
  c.qtd_novos,
  c.qtd_baixados,
  c.snapshot_ano,
  current_timestamp() AS _processamento_ts
FROM silver.cadastro_setor c;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fato_cadastro_municipio AS
SELECT
  CAST(concat(c.ano, lpad(c.mes, 2, '0')) AS INT) AS sk_tempo,
  c.categoria,
  c.cod_municipio_ibge,
  c.qtd_ativos,
  c.qtd_novos,
  c.qtd_baixados,
  c.snapshot_ano,
  current_timestamp() AS _processamento_ts
FROM silver.cadastro_municipio c;

##13. fato de PIB municipal
Grão: ano × município. Valores a preços correntes

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fato_pib_municipal AS
SELECT
  p.ano,
  p.cod_municipio_ibge,
  p.pib_reais,
  p.pib_mil_reais,
  current_timestamp() AS _processamento_ts
FROM silver.pib_municipal p;

##14. invariantes de negócio

In [0]:
%sql
CREATE OR REPLACE TABLE gold.relatorio_invariantes AS
-- 1. a Gold preserva a Silver
SELECT 'Reconciliação Silver x Gold' AS invariante, 'fato_icms_cnae' AS objeto,
       CAST((SELECT sum(valor_icms) FROM gold.fato_icms_cnae)
          - (SELECT sum(valor_icms) FROM silver.icms_cnae_subclasse) AS DOUBLE) AS valor,
       '0' AS esperado, current_timestamp() AS _execucao_ts
UNION ALL
SELECT 'Reconciliação Silver x Gold', 'fatos de desoneração',
       CAST((SELECT sum(valor_desonerado) FROM gold.fato_desoneracao_detalhada)
          + (SELECT sum(valor_desonerado) FROM gold.fato_desoneracao_agregada)
          - (SELECT sum(valor_desonerado) FROM silver.desoneracoes) AS DOUBLE),
       '0', current_timestamp()
UNION ALL
-- 2. o total estadual continua reproduzível a partir da Gold
SELECT 'Total estadual de 2023', 'fatos de desoneração x R$ 15 bi publicados',
       round((SELECT sum(f.valor_desonerado) FROM gold.fato_desoneracao_detalhada f
              JOIN gold.dim_beneficio b USING (imposto, tipo_beneficio, cod_beneficio)
              WHERE f.ano = 2023 AND b.integra_total_estadual)
           + (SELECT sum(f.valor_desonerado) FROM gold.fato_desoneracao_agregada f
              JOIN gold.dim_beneficio b USING (imposto, tipo_beneficio, cod_beneficio)
              WHERE f.ano = 2023 AND b.integra_total_estadual), 0) / 1e9,
       'aprox. 15', current_timestamp()
UNION ALL
-- 3. toda linha de fato encontra sua dimensão
SELECT 'Integridade referencial', 'fato_icms_cnae x dim_cnae',
       CAST((SELECT count(*) FROM gold.fato_icms_cnae f
             LEFT JOIN gold.dim_cnae d ON d.cnae_subclasse = f.cnae_subclasse
             WHERE d.cnae_subclasse IS NULL) AS DOUBLE),
       '0', current_timestamp()
UNION ALL
SELECT 'Integridade referencial', 'fato_icms_cnae x dim_tempo',
       CAST((SELECT count(*) FROM gold.fato_icms_cnae f
             LEFT JOIN gold.dim_tempo t ON t.sk_tempo = f.sk_tempo
             WHERE t.sk_tempo IS NULL) AS DOUBLE),
       '0', current_timestamp()
UNION ALL
SELECT 'Integridade referencial', 'fato_desoneracao_detalhada x dim_beneficio',
       CAST((SELECT count(*) FROM gold.fato_desoneracao_detalhada f
             LEFT JOIN gold.dim_beneficio b USING (imposto, tipo_beneficio, cod_beneficio)
             WHERE b.cod_beneficio IS NULL) AS DOUBLE),
       '0', current_timestamp()
UNION ALL
-- 4. a ponte é N:N e por isso as cadeias NÃO somam ao total
SELECT 'Ponte N:N', 'excedente do somatório por cadeia sobre o total (%)',
       round(100 * ((SELECT sum(f.valor_icms) FROM gold.fato_icms_cnae f
                     JOIN gold.ponte_cnae_cadeia p ON p.cnae_subclasse = f.cnae_subclasse)
                  - (SELECT sum(f.valor_icms) FROM gold.fato_icms_cnae f
                     JOIN gold.dim_cnae d ON d.cnae_subclasse = f.cnae_subclasse
                     WHERE d.tem_cadeia))
             / (SELECT sum(f.valor_icms) FROM gold.fato_icms_cnae f
                JOIN gold.dim_cnae d ON d.cnae_subclasse = f.cnae_subclasse
                WHERE d.tem_cadeia), 2),
       'maior que zero, por construção', current_timestamp();

In [0]:
%sql
SELECT * FROM gold.relatorio_invariantes;

##15. verificação final da camada

In [0]:
%sql
SELECT 'dim_tempo' AS tabela, count(*) AS linhas FROM gold.dim_tempo
UNION ALL SELECT 'dim_cnae', count(*) FROM gold.dim_cnae
UNION ALL SELECT 'dim_cadeia_produtiva', count(*) FROM gold.dim_cadeia_produtiva
UNION ALL SELECT 'dim_municipio', count(*) FROM gold.dim_municipio
UNION ALL SELECT 'dim_beneficio', count(*) FROM gold.dim_beneficio
UNION ALL SELECT 'dim_categoria_contribuinte', count(*) FROM gold.dim_categoria_contribuinte
UNION ALL SELECT 'ponte_cnae_cadeia', count(*) FROM gold.ponte_cnae_cadeia
UNION ALL SELECT 'fato_icms_cnae', count(*) FROM gold.fato_icms_cnae
UNION ALL SELECT 'fato_arrecadacao_municipio', count(*) FROM gold.fato_arrecadacao_municipio
UNION ALL SELECT 'fato_desoneracao_detalhada', count(*) FROM gold.fato_desoneracao_detalhada
UNION ALL SELECT 'fato_desoneracao_agregada', count(*) FROM gold.fato_desoneracao_agregada
UNION ALL SELECT 'fato_cadastro_setor', count(*) FROM gold.fato_cadastro_setor
UNION ALL SELECT 'fato_cadastro_municipio', count(*) FROM gold.fato_cadastro_municipio
UNION ALL SELECT 'fato_pib_municipal', count(*) FROM gold.fato_pib_municipal
ORDER BY tabela;